In [2]:
from dataclasses import dataclass
import datetime
import functools
import sys
import json
import os
from pathlib import Path

import numpy as np 
import scipy as sp
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import joblib
from sklearn.linear_model import HuberRegressor, LinearRegression


import nibabel as nib
from nilearn import masking


sys.path.append('/RAID1/jupytertmp/mcm/src')
import blood
import filenames

%load_ext autoreload
%autoreload 2

: 

In [2]:
container_name = 'cooltool'

container = ! docker ps | grep cooltool | awk '{print $1}'
if not container:

    container, = ! docker run \
        --name {container_name} \
        --gpus all \
        -it -d \
        -v /RAID1/jupytertmp/:/RAID1/jupytertmp \
        -v /RAID1/tmp:/RAID1/tmp\
        -v /RAID1/xnat:/RAID1/xnat:ro\
        valneurolab/supertool:v2
    
    print(f'started new container {container}')
else:
    container = container[0]
    print(f'acquired running container {container}')

container = container[:5]

started new container 51b49ffa2f67cd8b8b2821bfee762d7a22e5fd4ac1611169ecd5f2837cbf4d0f


In [2]:
@dataclass
class CONST:

    calibration:    float = 506 # mL^-1, obtained manually for each scanner
    calibration_adj: float = 1.849673202614379 # adjustment to the calibration factor for subjects after 23 
    gm_density:     float = 1045 * 1000 / 1e6 # g/mL, from https://itis.swiss/virtual-population/tissue-properties/database/density/
    wm_density:     float = 1041 * 1000 / 1e6 # g/mL
    LC:             float = 0.65 # Wu (2003) Molecular Imaging & Biology
    # LC:             float = 0.89 # no unit, Graham (2002) Joural of Nuclear Medicine

    # Solutions from Feng 1993
    A1, l1, A2, l2, A3, l3 = 767.3, -4.195, 27.43, -0.232, 29.13, -0.0165
    A1_sd, l1_sd, A2_sd, l2_sd, A3_sd, l3_sd = 360, 1.38, 7.48, 0.112, 5.22, 0.0076
    tau:            float = 0.905

    # RBC to plasma ratio for FDG - solutions from Phelps (1979)
    B:              float = 0.8 # y-axis intercept - no unit (?)
    K:              float = 0.0012 # slope of the line - min^-1

In [3]:
participants = pd.read_csv('/RAID1/jupytertmp/mcm/data/participants.csv')
participants.index = participants.id
participants['start time'] = pd.to_datetime(participants['start time'], format='%H:%M:%S')
participants['start time'] = participants['start time'].apply(lambda x: x.time())

In [4]:
for subject in participants.index:
    # pet = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{subject:03}/niftypet-recon/sub-s{subject:03}_desc-preproc_pet.nii.gz'
    pet = filenames.pet.format(id=subject)
    nframes = nib.load(pet).shape[-1]

    # recon_info = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{subject:03}/niftypet-recon/sub-s{subject:03}_pet.json'
    recon_info = filenames.recon_info.format(id=subject)
    with open(recon_info, 'r') as f:
        file = json.load(f)

    assert nframes == len(file['Time']['FrameTimes']['Values'])

In [5]:
participants.loc[17, 'offset'] = -25
participants.loc[20, 'offset'] = -23
participants.loc[23, 'offset'] = -30
participants.loc[28, 'offset'] = -30
participants.loc[29, 'offset'] = -25
participants.loc[30, 'offset'] = -33
participants.loc[31, 'offset'] = -37
participants.loc[32, 'offset'] = -25
participants.loc[33, 'offset'] = -30
participants.loc[34, 'offset'] = -30
participants.loc[35, 'offset'] = -40
participants.loc[37, 'offset'] = -30

In [4]:
quantification_subjects = [17, 20, 23, 28, 29, 30, 31, 32, 33, 34, 35, 37]
quant_participants = participants.loc[quantification_subjects]

In [7]:
(quant_participants.sex == 'F').sum()

np.int64(8)

In [5]:
(quant_participants.cohort - quant_participants.yob).describe()

count    12.000000
mean     31.916667
std       9.219134
min      23.000000
25%      25.750000
50%      27.500000
75%      38.750000
max      52.000000
dtype: float64

In [10]:
quant_participants.dose.std()

np.float64(8.14778274356672)

---

In [8]:
for id, subject in quant_participants.iterrows():

    print(subject.subject, end='\r')

    # aif_file = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/niftypet-recon/sub-s{id:03}_blood.crv'
    aif_file = filenames.aif.format(id=id)
    aif = pd.read_csv(
        aif_file,
        header=None, 
        sep='\s+', 
        names=['y', 'mo', 'd', 'h', 'm', 's', 'counts'], 
        usecols=range(7)
        )
    aif.index = aif.apply(lambda row: datetime.time(int(row.h), int(row.m), int(row.s)), axis=1)

    injection_start = subject['start time']
    injection_start_index = np.where(aif.index >= injection_start)[0][0]

    blood_tac = aif.loc[(injection_start <= aif.index), 'counts']
    blood_tac = np.array(blood_tac, dtype=np.float64) # Bq
    sigma = np.sqrt(blood_tac)
    background, = aif.loc[aif.index < injection_start, 'counts'].mode() # Bq
    time_min = np.linspace(0, (blood_tac.size - 1)/60, blood_tac.size) # min

    filtered = sp.signal.savgol_filter(blood_tac, window_length=30, polyorder=3)
    peaks, _ = sp.signal.find_peaks(filtered, height=np.quantile(filtered, 0.9))
    blood_delay = (peaks[0] + subject.offset) / 60 # mi

    p0 = [CONST.A1, CONST.l1, CONST.A2, CONST.l2, CONST.A3, CONST.l3]
    ydata = blood_tac
    xdata = time_min
    maxfev = 5000 #2500

    f = functools.partial(blood.feng, tau=blood_delay, background=background)
    popt, pcov = sp.optimize.curve_fit(
        f, xdata, ydata,
        p0=p0,
        sigma=sigma,
        absolute_sigma=True,
        full_output=False,
        maxfev=maxfev
    )

    A1, l1, A2, l2, A3, l3 = popt
    feng_fit = functools.partial(blood.feng, A1=A1, l1=l1, A2=A2, l2=l2, A3=A3, l3=l3, tau=blood_delay, background=background)
    p2b_fit = functools.partial(blood.plasma_to_blood_ratio, tau=blood_delay, hematocrit=subject.hematocrit, B=CONST.B, K=CONST.K)
    feng_plasma_fit = functools.partial(blood.feng_plasma, feng_fit=feng_fit, p2b_fit=p2b_fit, background=background, calibration=CONST.calibration)

    blood_params = {
        "background": background,
        "blood_delay": blood_delay,
        "time_min": time_min,
        "feng_fit": feng_fit,
        "p2b_fit": p2b_fit,
        "feng_plasma_fit": feng_plasma_fit
    }
    blood_params_file = filenames.blood_params.format(id=id)
    # blood_params_file = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/quantification/sub-s{id:03}_blood_params.joblib'
    joblib.dump(blood_params, blood_params_file)

<>:10: SyntaxWarning: invalid escape sequence '\s'
<>:10: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_981815/885291544.py:10: SyntaxWarning: invalid escape sequence '\s'
  sep='\s+',


/RAID1/jupytertmp/mcm/src/blood.py:21: RuntimeWarning: overflow encountered in exp
  lambda t: (A1 * (t - tau) - A2 - A3) * np.exp(l1 * (t - tau))
/RAID1/jupytertmp/mcm/src/blood.py:21: RuntimeWarning: overflow encountered in multiply
  lambda t: (A1 * (t - tau) - A2 - A3) * np.exp(l1 * (t - tau))


In [9]:
for id, subject in quant_participants.iterrows():

    print(subject.subject, end='\r')

    # PET stuff
    # recon_info = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/niftypet-recon/sub-s{id:03}_pet.json'
    recon_info = filenames.recon_info.format(id=id)
    with open(recon_info, 'r') as f:
        sidecar = json.load(f)
    frame_times = sidecar['Time']['FrameTimes']['Values']
    frame_duration = np.array([ft[1] - ft[0] for ft in frame_times])
    frame_reference_time = [np.mean(startend) for startend in frame_times]

    # pet = nib.load(f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/niftypet-recon/sub-s{id:03}_desc-preproc_pet.nii.gz')
    pet = nib.load(filenames.pet.format(id=id))
    # mask = nib.load(f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/niftypet-recon/sub-s{id:03}_desc-head_mask.nii.gz')
    mask = nib.load(filenames.pet_headmask.format(id=id))
    timeseries = masking.apply_mask(pet, mask)

    feng_plasma_fit = joblib.load(filenames.blood_params.format(id=id))['feng_plasma_fit']
    plasma_integral = np.array([
        sp.integrate.quad(feng_plasma_fit, 0, ft)[0] for ft in frame_reference_time
        ])

    patlak_x = plasma_integral / feng_plasma_fit(frame_reference_time)
    patlak_y = timeseries / feng_plasma_fit(frame_reference_time)[:, np.newaxis]

    pet_params = {
        "plasma_integral": plasma_integral,
        "frame_reference_time": frame_reference_time,
        "frame_duration": frame_duration,
        "patlak_x": patlak_x,
        "patlak_y": patlak_y
    }

    pet_params_file = filenames.pet_params.format(id=id)
    joblib.dump(pet_params, pet_params_file)

/tmp/ipykernel_981815/4194132087.py:25: RuntimeWarning: invalid value encountered in divide
  patlak_x = plasma_integral / feng_plasma_fit(frame_reference_time)
/tmp/ipykernel_981815/4194132087.py:26: RuntimeWarning: divide by zero encountered in divide
  patlak_y = timeseries / feng_plasma_fit(frame_reference_time)[:, np.newaxis]
/tmp/ipykernel_981815/4194132087.py:26: RuntimeWarning: invalid value encountered in divide
  patlak_y = timeseries / feng_plasma_fit(frame_reference_time)[:, np.newaxis]


In [10]:
for id, subject in quant_participants.iterrows():

    print(subject.subject, end='\r')

    mask = nib.load(filenames.pet_headmask.format(id=id))
    pet_params = joblib.load(filenames.pet_params.format(id=id))

    # Simple linear regression
    n = 5
    x = pet_params['patlak_x'][-n:]
    y = pet_params['patlak_y'][-n:, :]
    
    M = x[:, np.newaxis]**[0,1] # fit the intercept (constant value in the M matrix) and slope (x values in the M)
    p, res, rnk, s = sp.linalg.lstsq(M, y)

    V_B, Ki_voxel = p
    cmrglc_voxel = Ki_voxel * 100 * subject.Ca / (CONST.gm_density * CONST.LC)
    
    cmrglc_img = masking.unmask(cmrglc_voxel, mask)
    # cmrglc_file = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/quantification/sub-s{id:03}_quant-1mm_desc-5frames_cmrglc.nii.gz'
    cmrglc_file = filenames.cmrglc_5frames.format(id=id)
    nib.save(cmrglc_img, cmrglc_file)


    # With weighting
    n = 10
    x = pet_params['patlak_x'][-n:, np.newaxis]
    y = pet_params['patlak_y'][-n:, :]
    weights = pet_params['frame_duration'][-n:] / pet_params['frame_duration'][-n:].sum()
    model = LinearRegression(positive=True).fit(X=x, y=y, sample_weight=weights)

    Ki_voxel = model.coef_.flatten()
    cmrglc_voxel = Ki_voxel * 100 * subject.Ca / (CONST.gm_density * CONST.LC)
    cmrglc_img = masking.unmask(cmrglc_voxel, mask)
    # cmrglc_file = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/quantification/sub-s{id:03}_quant-1mm_desc-10frames_cmrglc.nii.gz'
    cmrglc_file = filenames.cmrglc_1mm.format(id=id)
    nib.save(cmrglc_img, cmrglc_file)


## Registrer the quantification to MNI ICMB 2009c nonlinear asymmetric 

In [11]:
mni = '/RAID1/jupytertmp/mcm/data/external/mni/mni_icbm152_nlin_asym_09c/mni_icbm152_t1_tal_nlin_asym_09c.nii.gz'
for id, subject in quant_participants.iterrows():
    
    pet2anat = filenames.pet2anat.format(id=id)
    anat2mni_affine = filenames.anat2mniICBM_aff.format(id=id)
    anat2mni_warp = filenames.anat2mniICBM_warp.format(id=id)
    input = filenames.cmrglc_1mm.format(id=id)
    output = filenames.cmrglc_mniICBM.format(id=id)

    ! docker exec {container} antsApplyTransforms \
        -d 3 \
        -i {input} \
        -r {mni} \
        -o {output} \
        -n BSpline \
        -t {anat2mni_warp} \
        -t {anat2mni_affine} \
        -t {pet2anat}

docker exec  cafcb antsApplyTransforms -d 3 -i /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/quantification/sub-s017_quant-cmrglc_res-1mm_desc-10frames_pet.nii.gz -r /RAID1/jupytertmp/mcm/data/external/mni/mni_icbm152_nlin_asym_09c/mni_icbm152_t1_tal_nlin_asym_09c.nii.gz -o /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/quantification/sub-s017_quant-cmrglc_res-1mm_desc-10frames_space-mniICBM2009cNlinAsym_pet.nii.gz -n BSpline -t /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/registration/sub-s017_anat2mniICBM2009cNlinAsym_1Warp.nii.gz -t /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/registration/sub-s017_anat2mniICBM2009cNlinAsym_0GenericAffine.mat -t /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/registration/sub-s017_pet2anat_0GenericAffine.mat
docker exec  cafcb antsApplyTransforms -d 3 -i /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s020/quantification/sub-s020_quant-cmrglc_res-1mm_desc-10frames_pet.nii.gz -r /RAID1/jupytertmp/mcm/data/external/mn

## Same for MNI152 1mm 

In [12]:
mni = '/RAID1/jupytertmp/mcm/data/external/mni/MNI152_T1_1mm.nii.gz'
for id, subject in quant_participants.iterrows():
    
    pet2anat = filenames.pet2anat.format(id=id)
    anat2mni_affine = filenames.anat2mni1mm_aff.format(id=id)
    anat2mni_warp = filenames.anat2mni1mm_warp.format(id=id)
    input = filenames.cmrglc_1mm.format(id=id)
    output = filenames.cmrglc_mni1mm.format(id=id)
 
    ! docker exec {container} antsApplyTransforms \
        -d 3 \
        -i {input} \
        -r {mni} \
        -o {output} \
        -n BSpline \
        -t {anat2mni_warp} \
        -t {anat2mni_affine} \
        -t {pet2anat}

docker exec  cafcb antsApplyTransforms -d 3 -i /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/quantification/sub-s017_quant-cmrglc_res-1mm_desc-10frames_pet.nii.gz -r /RAID1/jupytertmp/mcm/data/external/mni/MNI152_T1_1mm.nii.gz -o /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/quantification/sub-s017_quant-cmrglc_res-1mm_desc-10frames_space-mni1mm_pet.nii.gz -n BSpline -t /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/registration/sub-s017_anat2mni1mm_1Warp.nii.gz -t /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/registration/sub-s017_anat2mni1mm_0GenericAffine.mat -t /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/registration/sub-s017_pet2anat_0GenericAffine.mat
docker exec  cafcb antsApplyTransforms -d 3 -i /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s020/quantification/sub-s020_quant-cmrglc_res-1mm_desc-10frames_pet.nii.gz -r /RAID1/jupytertmp/mcm/data/external/mni/MNI152_T1_1mm.nii.gz -o /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s020/quantifi

## Concatenate the cmrglc maps in the time and average

In [ ]:
cmrglcs = '/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s*/quantification/sub-s*_quant-1mm_desc-10frames_space-mniICBM2009cNlinAsym_cmrglc.nii.gz'
map_4d = '/RAID1/jupytertmp/mcm/data/bids/derivatives/quant-1mm_desc-10frames_space-mniICBM2009cNlinAsym_cmrglcmap_4D.nii.gz'
map = '/RAID1/jupytertmp/mcm/data/bids/derivatives/quant-1mm_desc-10frames_space-mniICBM2009cNlinAsym_cmrglcmap.nii.gz'

# list_cmrglcs = ! ls {cmrglcs}
# list_cmrglcs = " ".join(list_cmrglcs)
# filenames.cmrglc_mniICBM.format(id='*') for all participants
! docker exec {container} fslmerge -t /RAID1/jupytertmp/mcm/data/bids/derivatives/quant-1mm_desc-10frames_space-mniICBM2009cNlinAsym_cmrglcmap_4D.nii.gz /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s*/quantification/sub-s*_quant-1mm_desc-10frames_space-mniICBM2009cNlinAsym_cmrglc.nii.gz
! docker exec {container} fslmaths /RAID1/jupytertmp/mcm/data/bids/derivatives/quant-1mm_desc-10frames_space-mniICBM2009cNlinAsym_cmrglcmap_4D.nii.gz -Tmean /RAID1/jupytertmp/mcm/data/bids/derivatives/quant-1mm_desc-10frames_space-mniICBM2009cNlinAsym_cmrglcmap.nii.gz

In [ ]:
fs = filenames.cmrglc_mniICBM.format(id='***')
! docker exec {container} ls {fs}

# Surface space

## Register cmrglc to anat space

In [17]:
for id, subject in quant_participants.iterrows():

    cmrglc = filenames.cmrglc_1mm.format(id=id)
    anat = filenames.t1.format(id=id)
    pet2anat = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/registration/sub-s{id:03}_pet2anat_0GenericAffine.mat'
    out = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/quantification/sub-s{id:03}_quant-cmrglc_res-1mm_desc-10frames_space-anat_pet.nii.gz'

    ! docker exec {container} antsApplyTransforms \
        -d 3 \
        -i {cmrglc} \
        -r {anat} \
        -o {out} \
        -t {pet2anat} \
        -n BSpline \
        -v 

docker exec  4e6ab antsApplyTransforms -d 3 -i /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/quantification/sub-s017_quant-cmrglc_res-1mm_desc-10frames_pet.nii.gz -r /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/ants-cortical-thickness/sub-s017_desc-preproc_T1w.nii.gz -o /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/quantification/sub-s017_quant-cmrglc_res-1mm_desc-10frames_space-anat_pet.nii.gz -t /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/registration/sub-s017_pet2anat_0GenericAffine.mat -n BSpline -v
Using double precision for computations.
Input scalar image: /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/quantification/sub-s017_quant-cmrglc_res-1mm_desc-10frames_pet.nii.gz
Reference image: /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/ants-cortical-thickness/sub-s017_desc-preproc_T1w.nii.gz
The composite transform comprises the following transforms (in order): 
  1. /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/registration/sub-s0

## Project to fsaverage space
### Make symbolic links for fsaverage5

In [20]:
src = '/usr/local/freesurfer/7.4.1/subjects/fsaverage5'

for id, subject in quant_participants.iterrows():

    dst = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/freesurfer/fsaverage5'
    ! ln -s {src} {dst}

### Do projection to fsaverage

In [6]:
for id, subject in quant_participants.iterrows():

    cmrglc_anat = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/quantification/sub-s{id:03}_quant-cmrglc_res-1mm_desc-10frames_space-anat_pet.nii.gz'
    fssubject = f'sub-s{id:03}'
    subjectdir = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/freesurfer'

    for space in ['fsaverage', 'fsaverage5']:
        for hemi in ['lh', 'rh']:

            out = f'/RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s{id:03}/quantification/sub-s{id:03}_quant-cmrglc_desc-10frames_hemi-{hemi}_space-{space}_pet.curv.gii'

            ! docker exec {container} mri_vol2surf \
                --mov {cmrglc_anat} \
                --regheader {fssubject} \
                --sd {subjectdir} \
                --hemi {hemi} \
                --o {out} \
                --trgsubject {space} \
                --projfrac-avg 0 1 0.2 \
                --cortex

docker exec  51b49 mri_vol2surf --mov /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/quantification/sub-s017_quant-cmrglc_res-1mm_desc-10frames_space-anat_pet.nii.gz --regheader sub-s017 --sd /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/freesurfer --hemi lh --o /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/quantification/sub-s017_quant-cmrglc_desc-10frames_hemi-lh_space-fsaverage_pet.curv.gii --trgsubject fsaverage --projfrac-avg 0 1 0.2 --cortex
srcvol = /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s017/quantification/sub-s017_quant-cmrglc_res-1mm_desc-10frames_space-anat_pet.nii.gz
srcreg unspecified
srcregold = 0
srcwarp unspecified
surf = white
hemi = lh
trgsubject = fsaverage
surfreg = sphere.reg
ProjFrac = 0.5
thickness = thickness
reshape = 0
interp = nearest
float2int = round
GetProjMax = 0
INFO: float2int code = 0
niiRead(): detected input as 64 bit double, reading in as 32 bit float
Done loading volume
Computing registration from header.
  Using /RAID1

### Compute the average

```bash
for space in fsaverage fsaverage5; do 
    for hemi in lh rh; do
        mri_concat \
            --i /RAID1/jupytertmp/mcm/data/bids/derivatives/sub-s*/quantification/sub-s*hemi-${hemi}*space-${space}_pet.mgh \
            --mean \
            --o /RAID1/jupytertmp/mcm/data/cmrglc-maps/quant-cmrglc_desc-10frames_hemi-${hemi}_space-${space}_pet.mgh
    done
done
```

### Smooth the final map for nice visualization

In [57]:
fwhm = 5 #mm

for space in ['fsaverage', 'fsaverage5']:
    for hemi in ['lh', 'rh']:
        orig = f'/RAID1/jupytertmp/mcm/data/cmrglc-maps/quant-cmrglc_desc-10frames_hemi-{hemi}_space-{space}_pet.mgh'
        smooth = f'/RAID1/jupytertmp/mcm/data/cmrglc-maps/quant-cmrglc_desc-10frames_fwhm-{fwhm}mm_hemi-{hemi}_space-{space}_pet.mgh'

        ! docker exec {container} mri_surf2surf \
            --s {space} \
            --hemi {hemi} \
            --sval {orig} \
            --tval {smooth} \
            --fwhm {fwhm}

docker exec  4e6ab mri_surf2surf --s fsaverage --hemi lh --sval /RAID1/jupytertmp/mcm/data/cmrglc-maps/quant-cmrglc_desc-10frames_hemi-lh_space-fsaverage_pet.mgh --tval /RAID1/jupytertmp/mcm/data/cmrglc-maps/quant-cmrglc_desc-10frames_fwhm-5mm_hemi-lh_space-fsaverage_pet.mgh --fwhm 5

7.4.1

setenv SUBJECTS_DIR /usr/local/freesurfer/7.4.1/subjects
cd /home/roman
mri_surf2surf --s fsaverage --hemi lh --sval /RAID1/jupytertmp/mcm/data/cmrglc-maps/quant-cmrglc_desc-10frames_hemi-lh_space-fsaverage_pet.mgh --tval /RAID1/jupytertmp/mcm/data/cmrglc-maps/quant-cmrglc_desc-10frames_fwhm-5mm_hemi-lh_space-fsaverage_pet.mgh --fwhm 5 

sysname  Linux
hostname 4e6abbb41f27
machine  x86_64
user     UNKNOWN
srcsubject = fsaverage
srcval     = /RAID1/jupytertmp/mcm/data/cmrglc-maps/quant-cmrglc_desc-10frames_hemi-lh_space-fsaverage_pet.mgh
srctype    = 
trgsubject = fsaverage
trgval     = /RAID1/jupytertmp/mcm/data/cmrglc-maps/quant-cmrglc_desc-10frames_fwhm-5mm_hemi-lh_space-fsaverage_pet.mgh
trgtyp

---